# RLVR for Tool-Calling with BFCL + llm-eval-kit

This notebook walks through training **Qwen 3 32B** to make better function calls using Reinforcement Fine-Tuning (RFT) on the [Berkeley Function Calling Leaderboard (BFCL)](https://gorilla.cs.berkeley.edu/leaderboard.html) dataset.

We use [llm-eval-kit](../../../../llm-eval-kit/) to handle the heavy lifting:
- Pull and format the BFCL dataset from HuggingFace
- Test the reward function locally with the built-in `tool_call` grader
- Deploy the reward function as an AWS Lambda
- Export training data in OpenAI-compatible format for Bedrock's fine-tuning APIs

## Prerequisites

- AWS credentials with permissions for IAM, Lambda, and Bedrock
- Bedrock model access for Qwen 3 32B
- llm-eval-kit installed: `uv pip install -e "path/to/llm-eval-kit[dev,datasets,deploy]"`

---
## 0. Install Dependencies

In [ ]:
%pip install -qU boto3 openai aws-bedrock-token-generator

# Install llm-eval-kit with all extras (adjust path as needed)
# %pip install -e "../../../../llm-eval-kit[dev,datasets,deploy]"

---
## 1. Configuration

In [ ]:
import boto3
import json
import time
import os

# ============== UPDATE THESE VALUES ==============
AWS_REGION = "us-west-2"
AWS_PROFILE = None  # Set to your profile name, or None for default
# =================================================

# BFCL configuration
HF_DATASET = "gorilla-llm/Berkeley-Function-Calling-Leaderboard"
BFCL_FILE = "BFCL_v3_exec_simple.json"
DATASET_NAME = "bfcl"
TOTAL_SAMPLES = None  # None = all, or set an integer to limit
LOCAL_DATA_DIR = "./tmp-data"

# Create session
session = (
    boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
    if AWS_PROFILE
    else boto3.Session(region_name=AWS_REGION)
)
AWS_ACCOUNT_ID = session.client("sts").get_caller_identity()["Account"]

# Resource names
LAMBDA_FUNCTION_NAME = f"{DATASET_NAME}-tool-call-reward"
LAMBDA_ROLE_NAME = f"{DATASET_NAME.upper()}-Lambda-Role"
REWARD_FUNCTION_FILE = "../../reward-functions/bfcl_tool_call_rew_func.py"

# Model — Qwen 3 32B via OpenAI-compatible API
MODEL_ID = "qwen.qwen3-32b-v1:0"
MANTLE_ENDPOINT = f"https://bedrock-mantle.{AWS_REGION}.api.aws"

# AWS clients
lambda_client = session.client("lambda")
iam_client = session.client("iam")

print(f"Account: {AWS_ACCOUNT_ID}")
print(f"Region:  {AWS_REGION}")
print(f"Model:   {MODEL_ID}")

---
## 2. Pull & Format Dataset

Use llm-eval-kit to pull BFCL from HuggingFace and export it in OpenAI-compatible RFT format.

Each training row will have:
- `messages`: The user prompt asking the model to make a function call
- `ground_truth`: The expected function call string for the reward function to check against

In [ ]:
from llm_eval_kit.datasets.loader import load_huggingface
from llm_eval_kit.datasets.formatter import export_rft_jsonl

# Pull BFCL from HuggingFace
dataset = load_huggingface(
    HF_DATASET,
    split="train",
    max_samples=TOTAL_SAMPLES,
    data_files=BFCL_FILE,
    prompt_key="question",
    ground_truth_key="ground_truth",
    id_key="id",
    response_key=None,
)
print(f"Loaded {len(dataset)} samples from HuggingFace")

# Preview a sample
s = dataset[0]
print(f"\nSample ID: {s.id}")
print(f"Messages:  {s.messages[0].content[:80]}...")
print(f"GT:        {str(s.ground_truth)[:80]}...")

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that responds to user requests by calling "
    "the appropriate function with the correct parameters. Output only the "
    "function call, e.g.: function_name(param1=value1, param2=value2)"
)

# Export as OpenAI-compatible RFT format with train/val/test split
split = export_rft_jsonl(
    dataset,
    output_dir=LOCAL_DATA_DIR,
    fmt="openai",  # OpenAI-compatible format for Qwen
    system_prompt=SYSTEM_PROMPT,
    train_ratio=0.8,
    val_ratio=0.1,
    seed=42,
)
print(f"Exported: {split.summary()}")
print(f"Train:    {split.train_path}")
print(f"Val:      {split.val_path}")

# Preview a formatted row
with open(split.train_path) as f:
    row = json.loads(f.readline())
print(f"\nFormatted row preview:")
print(json.dumps(row, indent=2)[:500])

---
## 3. Local Evaluation (Optional)

Before deploying, test the reward function locally using llm-eval-kit's built-in `tool_call` grader.

We simulate perfect responses by injecting ground truth as the assistant message, then run the grader pipeline.

In [ ]:
from llm_eval_kit.execution.pipeline import EvalPipeline
from llm_eval_kit.graders.registry import default_registry
from llm_eval_kit.models.messages import Message
import llm_eval_kit.graders  # registers built-ins

# Take a small subset for local testing
test_dataset = dataset[:20] if len(dataset) > 20 else dataset

# Create a copy with simulated perfect responses
from llm_eval_kit.models.datasets import EvalDataset, EvalSample

test_samples = []
for s in test_dataset:
    gt = s.ground_truth
    resp = gt[0] if isinstance(gt, list) else str(gt)
    test_samples.append(EvalSample(
        id=s.id,
        messages=s.messages + [Message(role="assistant", content=resp)],
        ground_truth=s.ground_truth,
        metadata=s.metadata,
    ))

grader = default_registry.get("tool_call")
report = EvalPipeline(grader, EvalDataset(test_samples)).run_with_report()
print(report.summary())
print("\n✓ Local evaluation passed — grader is working correctly")

---
## 4. Deploy the Reward Function

Use llm-eval-kit's `deploy_reward_function()` to deploy the standalone reward function as a Lambda.

This deploys just the single `.py` file — no pydantic or other dependencies needed, since the reward function is self-contained.

In [ ]:
# Create Lambda execution role
print("Creating Lambda execution role...")

lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}

try:
    response = iam_client.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(lambda_trust_policy),
        Description=f"Execution role for {DATASET_NAME} reward function",
    )
    lambda_role_arn = response["Role"]["Arn"]
    iam_client.attach_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    print(f"  ✓ Created role: {LAMBDA_ROLE_NAME}")
    print("  Waiting 10s for role propagation...")
    time.sleep(10)
except iam_client.exceptions.EntityAlreadyExistsException:
    lambda_role_arn = iam_client.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]
    print(f"  ✓ Using existing role: {LAMBDA_ROLE_NAME}")

In [ ]:
from llm_eval_kit.deploy.lambda_deploy import deploy_reward_function

# Deploy the standalone reward function
result = deploy_reward_function(
    source_file=REWARD_FUNCTION_FILE,
    function_name=LAMBDA_FUNCTION_NAME,
    role_arn=lambda_role_arn,
    runtime="python3.12",
    timeout=300,
    memory_size=512,
    region=AWS_REGION,
    profile=AWS_PROFILE,
)

lambda_arn = result["function_arn"]
print(f"\n✓ Lambda deployed: {lambda_arn}")

---
## 5. Test the Deployed Lambda

Verify the Lambda works with the Bedrock RFT batch contract before starting a training job.

In [ ]:
test_payload = [
    {
        "id": "test_correct",
        "messages": [
            {"role": "user", "content": "Calculate the area with base 10 and height 5"},
            {"role": "assistant", "content": "calculate_area(base=10, height=5)"},
        ],
        "metadata": {"ground_truth": ["calculate_area(base=10, height=5)"]},
    },
    {
        "id": "test_wrong",
        "messages": [
            {"role": "user", "content": "Calculate the area with base 10 and height 5"},
            {"role": "assistant", "content": "send_email(to='test@test.com')"},
        ],
        "metadata": {"ground_truth": ["calculate_area(base=10, height=5)"]},
    },
]

response = lambda_client.invoke(
    FunctionName=LAMBDA_FUNCTION_NAME,
    InvocationType="RequestResponse",
    Payload=json.dumps(test_payload),
)

results = json.loads(response["Payload"].read())
for r in results:
    score = r.get("aggregate_reward_score", 0)
    status = "✓" if score > 0.5 else "✗"
    print(f"  {status} {r['id']}: score={score}")

print("\n✓ Reward function working correctly")

---
## 6. Upload Training Data & Start RFT Job

Using the OpenAI-compatible API, we upload the training file directly (no S3 needed) and kick off the RFT job targeting Qwen 3 32B.

In [ ]:
from openai import OpenAI
from aws_bedrock_token_generator import provide_token

# Generate short-term Bedrock API key
bedrock_api_key = provide_token(region=AWS_REGION)

client = OpenAI(
    base_url=f"{MANTLE_ENDPOINT}/v1",
    api_key=bedrock_api_key,
)
print(f"✓ OpenAI client ready — {MANTLE_ENDPOINT}/v1")

In [ ]:
# Upload training file via OpenAI-compatible API
with open(split.train_path, "rb") as f:
    file_response = client.files.create(file=f, purpose="fine-tune")

training_file_id = file_response.id
print(f"✓ Training file uploaded: {training_file_id}")

In [ ]:
# Create RFT job
job_response = client.fine_tuning.jobs.create(
    model=MODEL_ID,
    training_file=training_file_id,
    extra_body={
        "method": {
            "type": "reinforcement",
            "reinforcement": {
                "grader": {
                    "type": "lambda",
                    "lambda": {"function": lambda_arn},
                },
                "hyperparameters": {
                    "n_epochs": 1,
                    "batch_size": 4,
                    "learning_rate_multiplier": 1.0,
                },
            },
        }
    },
)

job_id = job_response.id
print(f"✓ RFT job created: {job_id}")
print(f"  Model:  {MODEL_ID}")
print(f"  Status: {job_response.status}")

---
## 7. Monitor Training

Run these cells periodically to check progress. Watch `critic_rewards_mean` trending upward — that's the model learning to make better function calls.

In [ ]:
job_details = client.fine_tuning.jobs.retrieve(job_id)
print(f"Job:    {job_id}")
print(f"Status: {job_details.status}")

if job_details.status == "succeeded" and job_details.fine_tuned_model:
    print(f"\n✓ Training complete!")
    print(f"  Fine-tuned model: {job_details.fine_tuned_model}")
elif job_details.status == "failed":
    print(f"\n✗ Failed")
else:
    print("\nStill training... run this cell again to check.")

In [ ]:
# List training events
events = client.fine_tuning.jobs.list_events(
    fine_tuning_job_id=job_id, limit=50
)

metrics = [e.data for e in events.data if e.type == "metrics"]
if metrics:
    print(f"Training metrics ({len(metrics)} steps):")
    for m in metrics[-5:]:
        print(f"  Step {m['step']}/{m['total_steps']}: "
              f"reward={m['critic_rewards_mean']:.3f}, "
              f"entropy={m['actor_entropy']:.3f}")
else:
    print("No metrics yet — job may still be starting.")

---
## 8. Inference with Fine-Tuned Model

Once training completes, test the fine-tuned model.

In [ ]:
job_details = client.fine_tuning.jobs.retrieve(job_id)

if job_details.status == "succeeded" and job_details.fine_tuned_model:
    response = client.chat.completions.create(
        model=job_details.fine_tuned_model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Calculate the area of a triangle with base 10 and height 5"},
        ],
        max_tokens=256,
    )
    print(f"Model: {job_details.fine_tuned_model}")
    print(f"Response: {response.choices[0].message.content}")
else:
    print(f"Job status: {job_details.status} — wait for completion.")

---
## 9. Cleanup

In [ ]:
import shutil

if os.path.exists(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)
    print(f"✓ Cleaned up {LOCAL_DATA_DIR}")

---
## What We Built

The full loop using llm-eval-kit:

```python
# 1. Pull dataset
dataset = load_huggingface("gorilla-llm/Berkeley-Function-Calling-Leaderboard", ...)

# 2. Export to RFT format
split = export_rft_jsonl(dataset, output_dir, fmt="openai", system_prompt=...)

# 3. Test locally
report = EvalPipeline(grader, dataset).run_with_report()

# 4. Deploy reward function
result = deploy_reward_function(source_file, function_name, role_arn)

# 5. Upload & train via OpenAI-compatible API
client.files.create(...)
client.fine_tuning.jobs.create(...)
```

llm-eval-kit handled the dataset loading, formatting, splitting, and Lambda deployment — so the notebook focuses on the workflow rather than plumbing.

### Next Steps

- Try different BFCL subsets (`BFCL_v3_multiple.json`, `BFCL_v3_parallel.json`)
- Use `fmt="bedrock"` with `upload_to_s3()` for the Bedrock native API path (Nova models)
- Write custom graders with the `@grader` decorator for other evaluation tasks
- See the [llm-eval-kit docs](../../../../llm-eval-kit/src/llm_eval_kit/README.md) for the full framework reference